### Revision Notebook (ungraded)
Perform sentiment classification from movie reviews, using a Recurrent Neural Network (RNN)

Recurrent neural network is a type of neural network that is proven to work well with sequence data. Since text is actually a sequence of words, a recurrent neural network is an automatic choice to solve text-related problems. Here, we will use an LSTM (Long Short Term Memory network) which is a variant of RNN, to solve a movie reviews based sentiment classification problem. 

An LSTM unit consists of a cell, an input gate, an output gate and a forget gate. The cell remembers values over arbitrary time intervals and the three gates regulate the flow of information into and out of the cell.

LSTM networks are well-suited to classification based on time series data and deal well with the exploding and vanishing gradient problems that can be encountered when training traditional RNNs

<img style="-webkit-user-select: none;margin: auto;" src="https://miro.medium.com/max/1302/1*yr0820z7YNRcDCWGisLC4g.png" width="350" height="150">


<img style="-webkit-user-select: none;margin: auto;" src="https://miro.medium.com/max/1276/1*mvxPFvnDqj2jJrsjevD41A.png" width="350" height="150">







**Importing Required Libraries**

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

from numpy import array
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from keras.layers.core import Activation, Dropout, Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import GlobalMaxPooling1D
from keras.layers.embeddings import Embedding
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Dense, Embedding, LSTM, GRU

**Dataset**

In [ ]:
!wget https://cdn.talentsprint.com/aiml/Experiment_related_data/IMDB_Dataset.csv

**Dataset analysis**

In [ ]:
movie_reviews = pd.read_csv("/content/IMDB_Dataset.csv")

movie_reviews.isnull().values.any()

print (movie_reviews.shape)

print (movie_reviews.head())

In [ ]:
#Let us view one of the reviews
movie_reviews["review"][5]

In [ ]:
#View the distribution of positive and negative sentiments in the dataset
import pandas.util.testing as tm
import seaborn as sns
sns.countplot(x='sentiment', data=movie_reviews)

We can see above that there are an equal number of positive and negative sentiment reviews in the dataset

**Data Preprocessing**

Remove html tags, punctuations, special characters etc. from the review text

In [ ]:
#@title #####Data Preprocessing (double click to see the code)
def preprocess_text(sen):
    # Removing html tags
    sentence = remove_tags(sen)

    # Remove punctuations and numbers
    sentence = re.sub('[^a-zA-Z]', ' ', sentence)

    # Single character removal
    sentence = re.sub(r"\s+[a-zA-Z]\s+", ' ', sentence)

    # Removing multiple spaces
    sentence = re.sub(r'\s+', ' ', sentence)

    return sentence

#remove_tags() function removes html tags. 
#It replaces anything between opening and closing <> 
#with an empty space.
TAG_RE = re.compile(r'<[^>]+>')

def remove_tags(text):
    return TAG_RE.sub('', text)

In [ ]:
#Store the preprocessed reviews in a new list
X = []
sentences = list(movie_reviews['review'])
for sen in sentences:
    X.append(preprocess_text(sen))

In [ ]:
#Convert labels to integers

y = movie_reviews['sentiment']
y = np.array(list(map(lambda x: 1 if x=="positive" else 0, y)))

**Training and Testing**

In [ ]:
#Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

**Prepare the Embedding Layer**

The embedding layer converts our textual data into numeric data and is used as the first layer for the deep learning models in Keras

In [ ]:
#Tokenizer class from the keras.preprocessing.text module creates a word-to-index dictionary
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

The X_train set contains 40,000 lists of integers, each list corresponding to the sentences in a review. Set the maximum length of each list to 100 and add 0 padding to those lists that have a length <100, until they reach a length of 100

In [ ]:
# Find the vocabulary size and perform padding on both train and test set
vocab_size = len(tokenizer.word_index) + 1

maxlen = 100

X_train = pad_sequences(X_train, padding='post', maxlen=maxlen)
X_test = pad_sequences(X_test, padding='post', maxlen=maxlen)

In [ ]:
print ('number of unique words in the corpus:', vocab_size)

**Build LSTM Model for text classification**

In [ ]:


EMBEDDING_DIM = 100

print('Build model...')

model = Sequential()
model.add(Embedding(vocab_size, EMBEDDING_DIM, input_length=maxlen))
model.add(LSTM(units=32,  dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

# try using different optimizers and different optimizer configs
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print('Summary of the built model...')
print(model.summary())

**Training and Validation**

In [ ]:
history = model.fit(X_train, y_train, batch_size=128, epochs=3, verbose=2, validation_split=0.2)

**Testing**

In [ ]:
print('Testing...')
y_test = np.array(y_test)
#score = model.evaluate(X_test, y_test, verbose=1)
score, acc = model.evaluate(X_test, y_test, batch_size=128)

print('Test score:', score)
print('Test accuracy:', acc)

print("Accuracy: {0:.2%}".format(acc))

In [ ]:
#Let us test some  samples
test_sample_1 = "This movie is fantastic! I really like it because it is so good!"
test_sample_2 = "Good movie!"
test_sample_3 = "Maybe I like this movie."
test_sample_4 = "Not to my taste, will skip and watch another movie"
test_sample_5 = "if you like action, then this movie might be good for you."
test_sample_6 = "Bad movie!"
test_sample_7 = "Not a good movie!"
test_sample_8 = "This movie really sucks! Can I get my money back please?"
test_samples = [test_sample_1, test_sample_2, test_sample_3, test_sample_4, test_sample_5, test_sample_6, test_sample_7, test_sample_8]

test_samples_tokens = tokenizer.texts_to_sequences(test_samples)
test_samples_tokens_pad = pad_sequences(test_samples_tokens, maxlen=maxlen)

#predict
model.predict(x=test_samples_tokens_pad)

As we can see, the model has accurately predicted 7/8 of the test samples above as a positive sentiment (values close to 1) or a negative sentiment (values close to 0). 